[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [32]:
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]: #useful to speed-up generation during INFERENCE from target model
#to generate N tokens, we should run autoregressive loop, which is very slow
#instead, we use a smaller model to perform this autoregressive generation, obtaining the N candidates
#then the big model can evaluate them parallely like a single "pass" (each token containing all context of small model), deciding to reject or accept

#i have to stop after the first rejection because then the rejected token cannot eneter in the context to evaluate next once, since it was rejected
    K = draft_tokens.shape[0]
    res = []
    target_distrib = target_probs[torch.arange(K), draft_tokens] #(K,)
    draft_distrib = draft_probs[torch.arange(K), draft_tokens] #(K,)
    ratio = target_distrib / draft_distrib #(K,)

    for i in range(K):

      u = torch.rand(()).item()
      accept = u < min(1, ratio[i]) #()

      if not accept: #select one token, across all vocabulary, that has zero prob of being rejected, so 100% of being accepted (p(x) > q(x)), but always given the correct context
        distrib = target_probs[i] - draft_probs[i] #(V,)
        distrib = torch.maximum(torch.zeros_like(distrib), distrib)
        distrib = distrib / torch.sum(distrib, dim = -1, keepdim=True)

        chosen_tok = torch.multinomial(distrib, num_samples=1)
        res.append(chosen_tok.item())
        break

      res.append(draft_tokens[i].item())

    return res


In [33]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [2, 5, 1, 8]
Random draft: [2, 5, 1, 8]


In [34]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (5.1ms)
  ✅ [2/3] Output length bounded (2.6ms)
  ✅ [3/3] All tokens valid (35.5ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (43.2ms total)
  Progress saved. Run status() to see your dashboard.

